<a href="https://colab.research.google.com/github/ashoktamang002/Foundation-of-Data-Science/blob/main/Assessment%202%20Solution/Analytic%20Task%204/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIFA World Cup 2026: Goalkeeper Clean Sheets and Save Percentage
This notebook answers the analytical question:
Do goalkeepers who recorded at least one clean sheet have a significantly different average save percentage compared with goalkeepers who recorded no clean sheets?

# 1. Analytical Question and Hypotheses

Null hypothesis (H0): There is no significant difference in average save percentage between goalkeepers with at least one clean sheet and those with no clean sheets.

Alternative hypothesis (H1): There is a significant difference in average save percentage between the two groups.

In [12]:
import pandas as pd
import numpy as np
from scipy import stats

# 2. Data Collection
The data was manually collected into an Excel workbook from FBref's Player Goalkeeping 2026 World Cup table. The source data was obtained from the FBref website and organised into a workbook for this analysis.

Main FBref page: https://fbref.com/en/comps/1/keepers/World-Cup-Stats
The workbook contains goalkeeper-level data on appearances, shots on target faced, saves, save percentage, and clean sheets.

The required source fields are Player, MP, SoTA, Saves, and CS.

In [13]:
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
raw_data = pd.read_csv(file_name)
raw_data.head()

Saving World_Cup_2026_Player_Goalkeeping.csv to World_Cup_2026_Player_Goalkeeping (2).csv


,Player,MP,SoTA,Saves,CS
0,Yazeed Abulaila,3,16,8,0
1,Mahmoud Abunada,3,22,12,0
2,Alisson,5,18,14,2
3,Benjamin Asare,3,10,8,2
4,Lawrence Ati-Zigi,2,9,8,1


# 3. Data Wrangling
The raw table is reduced to the variables needed for the research question. Save percentage is calculated from saves and shots on target faced so that its definition is explicit.

In [14]:
raw_data.columns

Index(['Player', 'MP', 'SoTA', 'Saves', 'CS'], dtype='object')

In [15]:
df = raw_data[["Player", "MP", "SoTA", "Saves", "CS"]].copy()
df = df.rename(columns={
    "Player": "Goalkeeper",
    "MP": "Games_Played",
    "SoTA": "Shots_On_Target_Faced",
    "Saves": "Saves_Total",
    "CS": "Clean_Sheets"
})

numeric_columns = ["Games_Played", "Shots_On_Target_Faced", "Saves_Total", "Clean_Sheets"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

# A save percentage is not defined if a goalkeeper faced zero shots on target.
df["Save_Percentage"] = np.where(
    df["Shots_On_Target_Faced"] > 0,
    100 * df["Saves_Total"] / df["Shots_On_Target_Faced"],
    np.nan
)

df = df.dropna(subset=["Games_Played", "Clean_Sheets", "Save_Percentage"])
df = df[df["Games_Played"] > 0].copy()
df["Clean_Sheet_Group"] = np.where(
    df["Clean_Sheets"] >= 1,
    "At least one clean sheet",
    "No clean sheets"
)

df.head()

,Goalkeeper,Games_Played,Shots_On_Target_Faced,Saves_Total,Clean_Sheets,Save_Percentage,Clean_Sheet_Group
0,Yazeed Abulaila,3,16,8,0,50.000000,No clean sheets
1,Mahmoud Abunada,3,22,12,0,54.545455,No clean sheets
2,Alisson,5,18,14,2,77.777778,At least one clean sheet
3,Benjamin Asare,3,10,8,2,80.000000,At least one clean sheet
4,Lawrence Ati-Zigi,2,9,8,1,88.888889,At least one clean sheet


In [16]:
print("Final number of goalkeeper records:", len(df))
df.isnull().sum()

Final number of goalkeeper records: 59


,0
Goalkeeper,0
Games_Played,0
Shots_On_Target_Faced,0
Saves_Total,0
Clean_Sheets,0
Save_Percentage,0
Clean_Sheet_Group,0


# 4. Data Preparation and Sampling
Population: Goalkeepers who appeared at the FIFA World Cup 2026.
Sample: The goalkeeper records available in FBref's player goalkeeping table. This is an available, non-random sample, so results should be interpreted for the tournament dataset.
The two groups are naturally defined by clean sheets: at least one clean sheet versus no clean sheets.

# 5. Descriptive Statistics
The mean describes the average save percentage, the median gives the middle value, and the standard deviation indicates the spread within each group.

In [17]:
group_stats = df.groupby("Clean_Sheet_Group")["Save_Percentage"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
group_stats

,count,mean,median,std,min,max
Clean_Sheet_Group,,,,,,
At least one clean sheet,28,72.370451,72.077922,9.895211,50.0,88.888889
No clean sheets,31,50.694858,55.555556,19.369662,0.0,76.190476


In [18]:
clean_sheet_save_pct = df.loc[
    df["Clean_Sheet_Group"] == "At least one clean sheet", "Save_Percentage"
]
no_clean_sheet_save_pct = df.loc[
    df["Clean_Sheet_Group"] == "No clean sheets", "Save_Percentage"
]

# 6. 95% Confidence Intervals
A 95% confidence interval estimates a plausible range for the true mean save percentage of each group.

In [19]:
def mean_ci_95(values):
    n = values.count()
    mean = values.mean()
    standard_error = values.std(ddof=1) / np.sqrt(n)
    margin_error = stats.t.ppf(0.975, df=n - 1) * standard_error
    return mean, mean - margin_error, mean + margin_error

clean_mean, clean_ci_lower, clean_ci_upper = mean_ci_95(clean_sheet_save_pct)
no_clean_mean, no_clean_ci_lower, no_clean_ci_upper = mean_ci_95(no_clean_sheet_save_pct)

print("At least one clean sheet:")
print("Mean save percentage:", round(clean_mean, 2))
print("95% CI:", round(clean_ci_lower, 2), "to", round(clean_ci_upper, 2))
print()
print("No clean sheets:")
print("Mean save percentage:", round(no_clean_mean, 2))
print("95% CI:", round(no_clean_ci_lower, 2), "to", round(no_clean_ci_upper, 2))

At least one clean sheet:
Mean save percentage: 72.37
95% CI: 68.53 to 76.21

No clean sheets:
Mean save percentage: 50.69
95% CI: 43.59 to 57.8


# 7. Welch's Two-Sample t-Test
Welch's t-test is used because the two groups are independent and it does not assume equal variances.

In [20]:
t_statistic, p_value = stats.ttest_ind(
    clean_sheet_save_pct,
    no_clean_sheet_save_pct,
    equal_var=False
)

alpha = 0.05
print("t-statistic:", round(t_statistic, 3))
print("p-value:", round(p_value, 6))

t-statistic: 5.488
p-value: 2e-06


# 8. Interpretation and Final Conclusion

In [21]:
print("At-least-one-clean-sheet group mean save percentage:", round(clean_mean, 2))
print("At-least-one-clean-sheet group 95% CI:", round(clean_ci_lower, 2), "to", round(clean_ci_upper, 2))
print("No-clean-sheets group mean save percentage:", round(no_clean_mean, 2))
print("No-clean-sheets group 95% CI:", round(no_clean_ci_lower, 2), "to", round(no_clean_ci_upper, 2))
print("Welch t-test p-value:", round(p_value, 6))

if p_value < alpha:
    print("Decision: Reject H0.")
    print("Conclusion: Goalkeepers with at least one clean sheet have a significantly different average save percentage from those with no clean sheets.")
    print("In this dataset, the at-least-one-clean-sheet group has the higher average save percentage.")
else:
    print("Decision: Fail to reject H0.")
    print("Conclusion: There is insufficient evidence of a difference in average save percentage between the two groups.")

print("\nLimitation: This is an association, not proof that clean sheets cause a higher save percentage. Team defence, opponents, and match minutes may affect both measures.")

At-least-one-clean-sheet group mean save percentage: 72.37
At-least-one-clean-sheet group 95% CI: 68.53 to 76.21
No-clean-sheets group mean save percentage: 50.69
No-clean-sheets group 95% CI: 43.59 to 57.8
Welch t-test p-value: 2e-06
Decision: Reject H0.
Conclusion: Goalkeepers with at least one clean sheet have a significantly different average save percentage from those with no clean sheets.
In this dataset, the at-least-one-clean-sheet group has the higher average save percentage.

Limitation: This is an association, not proof that clean sheets cause a higher save percentage. Team defence, opponents, and match minutes may affect both measures.
